In [2]:
import pandas as pd

df = pd.read_csv('../../datasets/MX_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,5Z75FvRFiultmPFWHx5jQ7,7 Dias,"Gabito Ballesteros, Tito Double P",1,0,1,MX,2025-02-17,85,True,...,-3.749,1,0.0614,0.3860,0.000000,0.0839,0.577,111.913,3,Higher
1,78HEzDEs1QUnHB2DbxgC1s,Te Quería Ver,"Alemán, Neton Vega",2,1,1,MX,2025-02-17,82,False,...,-5.182,0,0.0681,0.1880,0.000017,0.0922,0.448,100.019,4,About_Average
2,0LTwdL5yZ6YOTEGUQPFuSN,ROSONES,Tito Double P,3,1,1,MX,2025-02-17,88,True,...,-5.939,1,0.0318,0.7040,0.000010,0.1170,0.604,120.129,3,Lower
3,7sd6zMrgGpEa7NkQm9TRrg,NADIE,Tito Double P,4,1,2,MX,2025-02-17,87,True,...,-4.710,1,0.1140,0.4650,0.000000,0.1200,0.526,92.604,4,About_Average
4,4eLDmhsJW3JoZTXCAozHor,Loco,Neton Vega,5,-3,45,MX,2025-02-17,61,False,...,-5.502,1,0.0686,0.0741,0.007680,0.1390,0.636,91.981,4,Lower


In [3]:
df.shape

(23806, 26)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity", 'daily_rank', 'daily_movement', 'weekly_movement'], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, [
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RandomForestRegressor(n_estimators=200, min_samples_split=2, min_samples_leaf=2, max_features=0.8, max_depth=30, n_jobs=4)
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_hist_grad_boosting_comparison.html


In [5]:
print("Random Forests Regression")
print("Mean Absolute Error: ", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error: ", mean_squared_error(y_test, y_pred))
print("Root Mean Squared Error: ", root_mean_squared_error(y_test, y_pred))
print("R2 Score: ", r2_score(y_test, y_pred))

for i in range(10):
    print(f"Predicted: {y_pred[i]}, Actual: {y_test.iloc[i]}")


Random Forests Regression
Mean Absolute Error:  1.7958006433337208
Mean Squared Error:  30.61929315028165
Root Mean Squared Error:  5.533470262889433
R2 Score:  0.6263780707173128
Predicted: 81.60145304028043, Actual: 81
Predicted: 94.08768282510262, Actual: 94
Predicted: 84.43086566293181, Actual: 82
Predicted: 92.38981484487735, Actual: 93
Predicted: 82.34822222222223, Actual: 83
Predicted: 80.8329519834742, Actual: 83
Predicted: 79.21928174603173, Actual: 82
Predicted: 82.1712559523809, Actual: 82
Predicted: 94.19471789460435, Actual: 94
Predicted: 82.07855555555558, Actual: 82


In [6]:
param_distributions = {
    'randomforestregressor__n_estimators': [50, 100, 200, 300],
    'randomforestregressor__max_depth': [5, 10, 20, 30, None],
    'randomforestregressor__min_samples_split': [2, 5, 10],
    'randomforestregressor__min_samples_leaf': [1, 2, 4],
    'randomforestregressor__max_features': ['sqrt', 'log2', 0.8]
}

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold

RFRGrid = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_iter=20,
    random_state=42,
    verbose=1
)

RFRGrid.fit(X_train, y_train)
print("Random Forests Regression with Grid Search")
print(RFRGrid.best_params_)
print(RFRGrid.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forests Regression with Grid Search
{'randomforestregressor__n_estimators': 100, 'randomforestregressor__min_samples_split': 2, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__max_features': 'log2', 'randomforestregressor__max_depth': None}
-32.082970015277745


Default = 25.49200082846408
{'randomforestregressor__n_estimators': 200, 'randomforestregressor__max_depth': 30} = -27.077155763516515
{'randomforestregressor__n_estimators': 200, 'randomforestregressor__min_samples_split': 2, 'randomforestregressor__min_samples_leaf': 2, 'randomforestregressor__max_features': 0.8, 'randomforestregressor__max_depth': 30}
-25.36150368665468, 1372 minutes


